In [3]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv(
    "/kaggle/input/datasets/bennygao2/sentiment-analysis-for-financial-news/all-data.csv",
    names=["sentiment", "text"],
    encoding="latin-1"
)


In [4]:
df["sentiment"].value_counts() #看 Positive / Neutral / Negative 有多少

sentiment
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64

In [5]:
##MACHINE LEARNING set up
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["sentiment"],
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]
)


vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

#Execute ML
model = LogisticRegression(max_iter=1000) #逻辑回归
model.fit(X_train_tfidf, y_train)

#Test Accuracy
y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred))

Accuracy: 0.7494845360824742
              precision    recall  f1-score   support

    negative       0.80      0.46      0.59       121
     neutral       0.76      0.92      0.83       576
    positive       0.71      0.52      0.60       273

    accuracy                           0.75       970
   macro avg       0.75      0.63      0.67       970
weighted avg       0.75      0.75      0.73       970



In [6]:
###FinBERT setup
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
print('Test cuda availability:',torch.cuda.is_available())
print('Model:',torch.cuda.get_device_name(0))

#FinBERT 

model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

####FinBERT test
text = df["text"].iloc[0]

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {key: value.to(device) for key, value in inputs.items()} #import to GPU

with torch.no_grad():
    outputs = model(**inputs) #FinBERT Analysing

####FinBERT Analysis 函数
def predict_finbert(texts):
    predictions = []

    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            outputs = model(**inputs)

        prediction = torch.argmax(outputs.logits, dim=1).item()

        predictions.append(prediction)

    return predictions

label_map = model.config.id2label

finbert_predictions = predict_finbert(X_test)

finbert_predictions = [
    label_map[pred]
    for pred in finbert_predictions
]

##FinBERT Result
from sklearn.metrics import accuracy_score, classification_report

finbert_accuracy = accuracy_score(
    y_test,
    finbert_predictions
)

print("FinBERT Accuracy:", finbert_accuracy)

print(
    classification_report(
        y_test,
        finbert_predictions
    )
)

Test cuda availability: True
Model: Tesla T4


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT Accuracy: 0.8721649484536083
              precision    recall  f1-score   support

    negative       0.76      0.97      0.85       121
     neutral       0.95      0.84      0.89       576
    positive       0.79      0.90      0.84       273

    accuracy                           0.87       970
   macro avg       0.84      0.90      0.86       970
weighted avg       0.88      0.87      0.87       970



In [7]:
###Qwen (Qwen2.5-1.5B-Instruct) Setup

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Qwen loaded!")

#Qwen Result
predictions = []

for i in range(len(X_test)):
    text = X_test.iloc[i]

    prompt = f"""
You are a financial sentiment classifier.

Classify the following financial news into exactly ONE of these three labels:
positive
neutral
negative

News:
{text}

Answer with only one label: positive, neutral, or negative.
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    text_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text_input,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False #Qwen was evaluated using deterministic greedy decoding (do_sample=False) to ensure reproducible predictions.
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip().lower()

    # 只保留我们允许的三个标签
    if "positive" in answer:
        prediction = "positive"
    elif "negative" in answer:
        prediction = "negative"
    elif "neutral" in answer:
        prediction = "neutral"
    else:
        prediction = "unknown"

    predictions.append(prediction)

from sklearn.metrics import accuracy_score, classification_report

qwen_accuracy = accuracy_score(y_test, predictions)

print("Qwen Accuracy:", qwen_accuracy)
print()

print(classification_report(
    y_test,
    predictions,
    labels=["negative", "neutral", "positive"]
))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded!
Qwen Accuracy: 0.7494845360824742

              precision    recall  f1-score   support

    negative       0.93      0.57      0.71       121
     neutral       0.78      0.81      0.79       576
    positive       0.65      0.70      0.67       273

    accuracy                           0.75       970
   macro avg       0.79      0.69      0.72       970
weighted avg       0.76      0.75      0.75       970



In [10]:
###Result and comparison

# Create results table

results = pd.DataFrame({
    "text": X_test.values,
    "true_label": y_test.values,
    "tfidf_prediction": y_pred,
    "finbert_prediction": finbert_predictions,
    "qwen_prediction": predictions
})

results["tfidf_correct"] = (
    results["tfidf_prediction"] == results["true_label"]
)

results["finbert_correct"] = (
    results["finbert_prediction"] == results["true_label"]
)

results["qwen_correct"] = (
    results["qwen_prediction"] == results["true_label"]
)


# 1. FinBERT 对，TF-IDF 和 Qwen 都错
finbert_only = results[
    (results["finbert_correct"]) &
    (~results["tfidf_correct"]) &
    (~results["qwen_correct"])
]

# 2. TF-IDF 对，FinBERT 和 Qwen 都错
tfidf_only = results[
    (results["tfidf_correct"]) &
    (~results["finbert_correct"]) &
    (~results["qwen_correct"])
]

# 3. Qwen 对，TF-IDF 和 FinBERT 都错
qwen_only = results[
    (results["qwen_correct"]) &
    (~results["tfidf_correct"]) &
    (~results["finbert_correct"])
]

# 4. 三个模型全部正确
all_correct = results[
    results["tfidf_correct"] &
    results["finbert_correct"] &
    results["qwen_correct"]
]

# 5. 三个模型全部错误
all_wrong = results[
    ~results["tfidf_correct"] &
    ~results["finbert_correct"] &
    ~results["qwen_correct"]
]

# 6. TF-IDF 和 FinBERT 对，Qwen 错
tfidf_finbert_only = results[
    results["tfidf_correct"] &
    results["finbert_correct"] &
    ~results["qwen_correct"]
]

# 7. FinBERT 和 Qwen 对，TF-IDF 错
finbert_qwen_only = results[
    results["finbert_correct"] &
    results["qwen_correct"] &
    ~results["tfidf_correct"]
]

# 8. TF-IDF 和 Qwen 对，FinBERT 错
tfidf_qwen_only = results[
    results["tfidf_correct"] &
    results["qwen_correct"] &
    ~results["finbert_correct"]
]

print("========== ERROR ANALYSIS SUMMARY ==========")

print("FinBERT only correct:", len(finbert_only))
print("TF-IDF only correct:", len(tfidf_only))
print("Qwen only correct:", len(qwen_only))

print("\nAll three correct:", len(all_correct))
print("All three wrong:", len(all_wrong))

print("\nTF-IDF + FinBERT correct, Qwen wrong:",
      len(tfidf_finbert_only))

print("FinBERT + Qwen correct, TF-IDF wrong:",
      len(finbert_qwen_only))

print("TF-IDF + Qwen correct, FinBERT wrong:",
      len(tfidf_qwen_only))


categories = {
    "FinBERT only correct": finbert_only,
    "TF-IDF only correct": tfidf_only,
    "Qwen only correct": qwen_only,
    "All three wrong": all_wrong,
    "TF-IDF + FinBERT correct, Qwen wrong": tfidf_finbert_only,
    "FinBERT + Qwen correct, TF-IDF wrong": finbert_qwen_only,
    "TF-IDF + Qwen correct, FinBERT wrong": tfidf_qwen_only
}

for name, data in categories.items():
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    if len(data) == 0:
        print("No cases.")
        continue

    display(
        data[
            ["text", "true_label",
             "tfidf_prediction",
             "finbert_prediction",
             "qwen_prediction"]
        ]
    )

========== ERROR ANALYSIS SUMMARY ==========
FinBERT only correct: 80
TF-IDF only correct: 23
Qwen only correct: 34

All three correct: 545
All three wrong: 24

TF-IDF + FinBERT correct, Qwen wrong: 116
FinBERT + Qwen correct, TF-IDF wrong: 105
TF-IDF + Qwen correct, FinBERT wrong: 43

FinBERT only correct


,text,true_label,tfidf_prediction,finbert_prediction,qwen_prediction
7,The diesel margin has remained high .,positive,neutral,positive,neutral
47,The measures result from weak demand in the sh...,negative,positive,negative,neutral
52,"Due to rapid expansion , the market share of T...",positive,neutral,positive,neutral
67,"According to Schmardin , Nordea will most like...",positive,neutral,positive,neutral
69,Affecto has helped SKAT to set up the platform...,neutral,positive,neutral,positive
...,...,...,...,...,...
889,stores 16 March 2010 - Finnish stationery and ...,negative,neutral,negative,neutral
899,Changes in the market situation and tougher pr...,negative,neutral,negative,neutral
911,ArcelorMittal Chief Executive Officer Lakshmi ...,negative,neutral,negative,neutral
922,Key shareholders of Finnish IT services provid...,positive,neutral,positive,neutral



TF-IDF only correct


,text,true_label,tfidf_prediction,finbert_prediction,qwen_prediction
19,He said : `` It is for sale again and we will ...,neutral,neutral,positive,positive
21,The ore body is sufficient to support anticipa...,neutral,neutral,positive,positive
60,W+Ærtsil+Æ 's aim is to serve the rapidly grow...,neutral,neutral,positive,positive
81,"shock phase ' , consumers have once again star...",neutral,neutral,positive,positive
84,wins 98 % acceptance 23 December 2009 - Finnis...,neutral,neutral,positive,positive
131,The new units should become one of the largest...,neutral,neutral,positive,positive
149,"Together , Yahoo and Nokia said they will cont...",neutral,neutral,positive,positive
181,The board further said the company omitted to ...,neutral,neutral,negative,negative
195,Our purchase to pay solutions enable companies...,neutral,neutral,positive,positive
221,Finnish home decoration and arts and crafts re...,neutral,neutral,negative,negative



Qwen only correct


...

,text,true_label,tfidf_prediction,finbert_prediction,qwen_prediction
3,Finnish Rautaruukki has been awarded a contrac...,positive,positive,positive,neutral
8,"Operating profit , excluding non-recurring ite...",negative,negative,negative,neutral
14,Finnish aluminium products manufacturer Nordic...,neutral,neutral,neutral,positive
20,The first charging stations will be made avail...,neutral,neutral,neutral,positive
30,"Stora Enso , a global paper , packaging and wo...",neutral,neutral,neutral,positive
...,...,...,...,...,...
931,There has been some recovery of the base metal...,positive,positive,positive,neutral
935,The company has delivered the technical infras...,neutral,neutral,neutral,positive
943,The company will propose a dividend of EUR0 .1...,neutral,neutral,neutral,positive
952,Under the agreement Benefon 's forthcoming ran...,neutral,neutral,neutral,positive



FinBERT + Qwen correct, TF-IDF wrong


,text,true_label,tfidf_prediction,finbert_prediction,qwen_prediction
6,The subscriptions increase Cargotec 's share c...,positive,neutral,positive,positive
15,"` For Nordea , moving into the new headquarter...",positive,neutral,positive,positive
36,"After non-recurring items of EUR 177mn , profi...",neutral,negative,neutral,neutral
41,Combining this deep domain expertise with our ...,positive,neutral,positive,positive
43,Combining the two producers will create a stro...,positive,neutral,positive,positive
...,...,...,...,...,...
918,Finnish financial solutions developer Basware ...,positive,neutral,positive,positive
933,"( ADPnews ) - Oct 21 , 2009 - Finland-based IT...",negative,positive,negative,negative
946,FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is ag...,positive,neutral,positive,positive
965,Nokia will continue to invest in future develo...,positive,neutral,positive,positive



TF-IDF + Qwen correct, FinBERT wrong


,text,true_label,tfidf_prediction,finbert_prediction,qwen_prediction
24,Neste Oil extended yesterday 's gains and put ...,neutral,neutral,positive,neutral
42,Metso Foundries Jyvaskyla Oy will discontinue ...,neutral,neutral,negative,neutral
53,The acquisition is part of Suomen Helasto 's s...,neutral,neutral,positive,neutral
78,"According to Olvi , the company is already neg...",neutral,neutral,positive,neutral
88,The company will also shut one paper machine i...,neutral,neutral,negative,neutral
125,With the second phase of the new Innova buildi...,neutral,neutral,positive,neutral
164,On the basis of competitive bidding OOO Kitai ...,neutral,neutral,positive,neutral
188,"Loudeye Corp. , up $ 2.56 at $ 4.33 Nokia Corp...",neutral,neutral,negative,neutral
209,`` This transaction supports Huhtamaki 's stra...,neutral,neutral,positive,neutral
271,Nokia has inaugurated its manufacturing plant ...,neutral,neutral,positive,neutral
